In [22]:
import wget
import torch
import numpy
import torch.nn as nn
from torch.nn import functional as F

In [4]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
output_file = wget.download(url)

100% [..........................................................................] 1115394 / 1115394

In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [23]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [24]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [25]:
# encode all of input text and store in tensor
data = torch.tensor(encode(text), dtype=torch.long)

In [26]:
# create train and validation split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [28]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [32]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix =  torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} te target: {target}")

when input is [24] te target: 43
when input is [24, 43] te target: 58
when input is [24, 43, 58] te target: 5
when input is [24, 43, 58, 5] te target: 57
when input is [24, 43, 58, 5, 57] te target: 1
when input is [24, 43, 58, 5, 57, 1] te target: 46
when input is [24, 43, 58, 5, 57, 1, 46] te target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] te target: 39
when input is [44] te target: 53
when input is [44, 53] te target: 56
when input is [44, 53, 56] te target: 1
when input is [44, 53, 56, 1] te target: 58
when input is [44, 53, 56, 1, 58] te target: 46
when input is [44, 53, 56, 1, 58, 46] te target: 39
when input is [44, 53, 56, 1, 58, 46, 39] te target: 58
when input is [44, 53, 56, 1, 58, 46, 39, 58] te target: 1
when input is [52] te target: 58
when input is [52, 58] te target: 1
when input is [52, 58, 1] te target: 58
when input is [52, 58, 1, 58] te target: 46
when input is [52, 58, 1, 58, 46] te target: 39
when input is [52, 58, 1, 58, 46, 39] te target: 58
when input i

In [37]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8231, grad_fn=<NllLossBackward0>)

fW.xwni,&HVGYHy3MP;yMDsxZ.!z,v-x$NknAukgY&SYW$Ggs;IYaPc-cTqC !R3fF!

Zg-$wcfCA?WuF:dP-ASfF?:koZHSsJA


In [38]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [44]:
# Training loop
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.457082986831665


In [45]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


W:
NGOLI bed hy be thy blave oupthe d,
D wigonrathabl, sh akes y hyeakeerark tingombe me eioforitot 


In [46]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])